# FahMai PaddleOCR Pipeline

Single notebook entry point for the production OCR flow.

- Uses PaddleOCR checkpoints and deterministic layout rules.
- Does not use full-page generative OCR.
- Rebuilds `ocr_outputs/submission_OCR_image_reviewed.csv` from checkpoints in this notebook.
- Optional full OCR rerun calls the Paddle production runner scripts when public renders are available.


In [ ]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import re
import subprocess

csv.field_size_limit(2_147_483_647)

SHORT_ROOT = Path(r"C:\\fahmai_ocr_data")

def find_ocr_root():
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd.parent.parent, SHORT_ROOT]
    for candidate in candidates:
        if (candidate / "submission_template_OCR.csv").is_file() and (candidate / "ocr_outputs").is_dir():
            return candidate
    raise FileNotFoundError("Cannot find OCR root. Open the notebook from the package root or notebooks folder.")

OCR_ROOT = find_ocr_root()
PIPELINE = OCR_ROOT / "ocr_pipeline"
OUTPUTS = OCR_ROOT / "ocr_outputs"
FINAL_CSV = OUTPUTS / "submission_OCR_image_reviewed.csv"
FAST_PARTIAL = OUTPUTS / "submission_OCR_fast_partial.csv"
ANNOTATIONS = OUTPUTS / "audits" / "image_grounded" / "manual_ground_truth_annotations.csv"
TEMPLATE = OCR_ROOT / "submission_template_OCR.csv"
CPU_PYTHON = Path(r"C:\\fahmai_paddle\\Scripts\\python.exe")

CHECKPOINT_DIRS = [
    OUTPUTS / "fast_dense_bank",
    OUTPUTS / "fast_compact_bank",
    OUTPUTS / "fast_bbl_bank",
    OUTPUTS / "fast_sparse_bank",
    OUTPUTS / "fast_fixed_nonbank",
    OUTPUTS / "fast_general_documents",
]
CHECKPOINT_VERSIONS = {
    "compact_scb_operating": 5,
    "compact_bbl_operating": 1,
    "sparse_scb_direct": 2,
    "sparse_bbl_direct": 2,
    "fixed_receipt": 1,
    "fixed_vendor_invoice": 1,
    "fixed_warranty_form": 1,
    "fast_general_document": 1,
}
TX_FIELDS = ["business_event_date", "transaction_type", "amount_thb", "balance_after_thb", "description", "account_id"]
TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(?::\d{2})?\s*$")
TIME_PATTERN = re.compile(r"\d{1,2}:\d{2}")

print("OCR_ROOT:", OCR_ROOT)
print("Template exists:", TEMPLATE.exists())
print("Annotation exists:", ANNOTATIONS.exists())
print("Checkpoint dirs:", {p.name: p.exists() for p in CHECKPOINT_DIRS})


In [ ]:
def read_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def write_submission(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["artifact_id", "pred_json"])
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def is_current_checkpoint(checkpoint):
    expected = CHECKPOINT_VERSIONS.get(checkpoint.get("layout"))
    return expected is None or checkpoint.get("checkpoint_version") == expected

def fill_dense_bank(prediction, checkpoint):
    account_id = checkpoint["account_id"]
    layout = checkpoint.get("layout")
    is_scb = layout in {"compact_scb_operating", "sparse_scb_direct"}
    is_bbl = layout in {"compact_bbl_operating", "sparse_bbl_direct"}
    header = {
        "L0_account_id": account_id,
        "L0_bank": "BBL" if is_bbl else "SCB" if is_scb else "KBANK",
        "L0_account_number": checkpoint.get("account_number", ""),
        "L0_account_role": "SAVINGS" if is_scb or is_bbl else "SAVING" if account_id.startswith("KBANK-") else "SAVINGS",
        "L0_currency": "THB",
    }
    for key, value in header.items():
        if key in prediction:
            prediction[key] = value
    for row in checkpoint.get("rows", []):
        for field in TX_FIELDS:
            key = f"L{row['level']}_{row['slot_id']}_{field}"
            if key in prediction:
                prediction[key] = row.get(field, "")
    for key, value in checkpoint.get("generic_values", {}).items():
        if key in prediction:
            prediction[key] = value

def fill_checkpoint(prediction, checkpoint):
    if "prediction" in checkpoint:
        for key, value in checkpoint.get("prediction", {}).items():
            if key in prediction:
                prediction[key] = value
    else:
        fill_dense_bank(prediction, checkpoint)

def normalize_bank_business_dates(prediction):
    normalized_count = 0
    for key, value in prediction.items():
        if key.endswith("_business_event_date") and value:
            date_only = TIME_SUFFIX.sub("", str(value)).strip()
            if date_only != value:
                prediction[key] = date_only
                normalized_count += 1
    return normalized_count


In [ ]:
def build_fast_partial_submission():
    template_rows = read_csv(TEMPLATE)
    checkpoints = {}
    for checkpoint_dir in CHECKPOINT_DIRS:
        for path in checkpoint_dir.glob("*.json"):
            checkpoints[path.stem] = json.loads(path.read_text(encoding="utf-8"))

    output_rows = []
    filled_by_prefix = Counter()
    total_by_prefix = Counter()
    used_checkpoints = checkpoint_rows = fallback_rows = normalized_dates = 0

    for template_row in template_rows:
        artifact_id = template_row["artifact_id"]
        prediction = json.loads(template_row["pred_json"])
        checkpoint = checkpoints.get(artifact_id)
        if checkpoint and not checkpoint.get("errors") and is_current_checkpoint(checkpoint):
            fill_checkpoint(prediction, checkpoint)
            used_checkpoints += 1
            checkpoint_rows += len(checkpoint.get("rows", []))
            fallback_rows += int(checkpoint.get("fallback_rows", checkpoint.get("fallback_count", 0)))
        if artifact_id.startswith("BS-"):
            normalized_dates += normalize_bank_business_dates(prediction)
        prefix = artifact_id.split("-", 1)[0]
        total_by_prefix[prefix] += len(prediction)
        filled_by_prefix[prefix] += sum(value != "" for value in prediction.values())
        output_rows.append({"artifact_id": artifact_id, "pred_json": json.dumps(prediction, ensure_ascii=False)})

    write_submission(FAST_PARTIAL, output_rows)
    audit = {
        "output": str(FAST_PARTIAL),
        "artifacts": len(output_rows),
        "checkpoints_found": len(checkpoints),
        "checkpoints_used": used_checkpoints,
        "checkpoint_rows": checkpoint_rows,
        "fallback_rows": fallback_rows,
        "normalized_bank_business_event_dates": normalized_dates,
        "filled_fields_by_prefix": dict(sorted(filled_by_prefix.items())),
        "total_fields_by_prefix": dict(sorted(total_by_prefix.items())),
    }
    (OUTPUTS / "submission_OCR_fast_partial.audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    return audit

audit = build_fast_partial_submission()
print(json.dumps(audit, ensure_ascii=False, indent=2))


In [ ]:
def load_submission(path):
    rows = read_csv(path)
    return [row["artifact_id"] for row in rows], {row["artifact_id"]: json.loads(row["pred_json"]) for row in rows}

def apply_verified_image_annotations(skip_render_path_check=True):
    template_order, template = load_submission(TEMPLATE)
    base_order, candidate = load_submission(FAST_PARTIAL)
    if base_order != template_order:
        raise RuntimeError("Base submission artifact order differs from public template.")
    annotations = read_csv(ANNOTATIONS)
    applied = {}
    ignored_status = 0
    errors = []
    for index, row in enumerate(annotations, start=2):
        if row.get("verification_status", "").strip() != "verified_from_image":
            ignored_status += 1
            continue
        artifact_id = row.get("artifact_id", "").strip()
        field_path = row.get("field_path", "").strip()
        if artifact_id not in template:
            errors.append(f"line {index}: unknown artifact_id={artifact_id!r}")
            continue
        if field_path not in template[artifact_id]:
            errors.append(f"line {index}: unknown field_path={field_path!r} artifact_id={artifact_id!r}")
            continue
        key = (artifact_id, field_path)
        if key in applied and applied[key]["verified_value"] != row.get("verified_value", ""):
            errors.append(f"line {index}: conflicting verified values for {artifact_id}.{field_path}")
            continue
        applied[key] = row
    if errors:
        raise RuntimeError("Annotation validation failed:\n- " + "\n- ".join(errors[:50]))

    changed = unchanged = intentional_blank = 0
    by_prefix = Counter()
    for (artifact_id, field_path), row in applied.items():
        value = row.get("verified_value", "")
        previous = str(candidate[artifact_id].get(field_path, ""))
        candidate[artifact_id][field_path] = value
        changed += previous != value
        unchanged += previous == value
        intentional_blank += value == ""
        by_prefix[artifact_id.split("-", 1)[0]] += 1

    rows = [{"artifact_id": artifact_id, "pred_json": json.dumps(candidate[artifact_id], ensure_ascii=False)} for artifact_id in template_order]
    write_submission(FINAL_CSV, rows)
    audit = {
        "base_submission": str(FAST_PARTIAL),
        "annotations": str(ANNOTATIONS),
        "output": str(FINAL_CSV),
        "safety": {
            "required_status": "verified_from_image",
            "public_render_path_required": not skip_render_path_check,
            "peer_submission_read": False,
            "grader_only_provenance_read": False,
            "enterprise_tables_read": False,
        },
        "annotation_rows": len(annotations),
        "verified_rows_applied": len(applied),
        "ignored_unverified_status_rows": ignored_status,
        "changed_fields": changed,
        "unchanged_fields": unchanged,
        "intentional_blank_fields": intentional_blank,
        "applied_by_prefix": dict(sorted(by_prefix.items())),
    }
    FINAL_CSV.with_suffix(".audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    return audit

annotation_audit = apply_verified_image_annotations(skip_render_path_check=True)
print(json.dumps(annotation_audit, ensure_ascii=False, indent=2))


In [ ]:
def validate_submission(path=FINAL_CSV):
    template_rows = read_csv(TEMPLATE)
    submission_rows = read_csv(path)
    expected_ids = [row["artifact_id"] for row in template_rows]
    actual_ids = [row["artifact_id"] for row in submission_rows]
    template_by_id = {row["artifact_id"]: json.loads(row["pred_json"]) for row in template_rows}
    empty = invalid = schema_mismatch = missing_keys = extra_keys = bs_time = total_fields = 0
    for row in submission_rows:
        value = row.get("pred_json", "")
        if not value:
            empty += 1
            continue
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            invalid += 1
            continue
        if not isinstance(parsed, dict):
            invalid += 1
            continue
        total_fields += len(parsed)
        expected = template_by_id.get(row["artifact_id"])
        if expected is not None and set(parsed) != set(expected):
            schema_mismatch += 1
            missing_keys += len(set(expected) - set(parsed))
            extra_keys += len(set(parsed) - set(expected))
        if row["artifact_id"].startswith("BS-"):
            bs_time += sum(bool(TIME_PATTERN.search(str(v))) for k, v in parsed.items() if k.endswith("_business_event_date") and v)
    problems = []
    if actual_ids != expected_ids:
        problems.append("artifact_id sequence differs from template")
    if empty or invalid or schema_mismatch or bs_time:
        problems.append(f"empty={empty} invalid={invalid} schema_mismatch={schema_mismatch} bs_time={bs_time}")
    print(f"rows={len(submission_rows)} expected_rows={len(template_rows)}")
    print(f"empty={empty} invalid_json={invalid} schema_mismatch={schema_mismatch} missing_keys={missing_keys} extra_keys={extra_keys} total_fields={total_fields} bs_business_event_dates_with_time={bs_time}")
    if problems:
        raise RuntimeError("validation=FAIL " + "; ".join(problems))
    print("validation=PASS")
    print("final_csv=", FINAL_CSV)
    print("sha256=", sha256(FINAL_CSV))

validate_submission(FINAL_CSV)


## Optional Full PaddleOCR Rerun

Set `RUN_FULL_PADDLE_OCR = True` only when the public render dataset exists and you want to rerun OCR extraction. This calls the production Paddle runner scripts from the notebook.

In [ ]:
RUN_FULL_PADDLE_OCR = False

def run_ps(script_name, timeout=None):
    command = ["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File", str(PIPELINE / script_name)]
    print("$", " ".join(command))
    result = subprocess.run(command, cwd=str(OCR_ROOT), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {script_name}")
    return result

if RUN_FULL_PADDLE_OCR:
    run_ps("run_exam_fast_ocr.ps1", timeout=7200)
else:
    print("Skipped full PaddleOCR rerun. Existing checkpoints were used to rebuild the final CSV.")
